# 11.10 · Streaming deep agents to a UI

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 1.5 h |
| 📋 Prerequisites | Lessons 11.05 and 11.08; Lesson 02.04 (streaming). |
| 🔑 Keys | none for 🧪 cells · optional 💸 cell ≈ 6–10K tokens (off by default) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Stream** a deep agent with the stable v2 API and tell coordinator and subagent events apart by namespace.
2. **Use** event streaming (v3): typed projections for messages, tool calls and subagents.
3. **Map** raw events to the small set of UI events a front end needs, and send them as Server-Sent Events.
4. **Handle** interrupts in a streaming UI.

## 📚 Official reading (verified 2026-10-16, `deepagents` 0.7.26, `langgraph` 1.2.15)

| Page | Read it for |
|---|---|
| [Event streaming (Deep Agents)](https://docs.langchain.com/oss/python/deepagents/event-streaming) | `stream.subagents`, handles, `interleave`, raw events |
| [Streaming (Deep Agents)](https://docs.langchain.com/oss/python/deepagents/streaming) | v2: `subgraphs=True`, namespaces, subagent progress |
| [Event streaming (LangChain)](https://docs.langchain.com/oss/python/langchain/event-streaming) | The general projections: messages, tool calls, values |

> 🧪 **Status (2026-10-16):** the docs recommend event streaming (v3) for new apps, but LangGraph 1.2.15 still emits "The v3 streaming protocol on Pregel is experimental". This lesson teaches both: v3 for its ergonomics, v2 as the stable fallback. Pin versions and re-test before upgrading.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "deepagents>=0.7.23,<0.8" "langchain-google-genai>=4.4.1,<5" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("deepagents", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-streaming'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Helpers: a coordinator that delegates to a "technical" subagent (scripted, no tokens).
import json
import warnings

from langchain.tools import tool
from langchain_core.messages import AIMessage

from mastery.deep import build_deep_agent, run_stats
from mastery.testing import ScriptedChatModel

# The v3 protocol is marked experimental: LangGraph emits a LangChainBetaWarning each run. We silence it HERE only.
warnings.filterwarnings("ignore", message=".*v3 streaming protocol.*")


def calls(*items) -> AIMessage:
    return AIMessage("", tool_calls=[{"name": n, "args": a, "id": i} for n, a, i in items])


@tool
def check_service_status(service: str) -> str:
    """Return the current status of a service."""
    return f"{service}: operational"


def make_agent():
    sub = ScriptedChatModel(responses=[calls(("check_service_status", {"service": "api"}, "s1")),
                                       AIMessage("The API is operational.")])
    coordinator = ScriptedChatModel(responses=[
        calls(("task", {"description": "Check the API status", "subagent_type": "technical"}, "t1")),
        AIMessage("All good: the API is operational.")])
    technical = {"name": "technical", "description": "Checks service status.", "system_prompt": "You check status.",
                 "tools": [check_service_status], "model": sub}
    return build_deep_agent(coordinator, subagents=[technical], fallbacks=False)


QUESTION = {"messages": [{"role": "user", "content": "Is the API up?"}]}

---
## Part 1 · v2: updates with namespaces

In [ ]:
# 🧪 The stable way (v2): updates per node, with a namespace telling you WHICH agent produced them.
for chunk in make_agent().stream(QUESTION, stream_mode="updates", subgraphs=True, version="v2"):
    nodes = [n for n in chunk["data"] if "Middleware" not in n]     # hide middleware hooks
    if nodes:
        who = "coordinator" if not chunk["ns"] else f"subagent {chunk['ns'][0][:14]}…"
        print(f"{who:<28} {nodes}")

**You should now see** coordinator `model`, then the subagent's `model`/`tools`/`model` under a `tools:…` namespace, then the coordinator's `tools` and final `model`. An empty namespace means the coordinator; anything else is a subagent run inside the `task` tool.

## Part 2 · v3: projections and subagent handles

In [ ]:
# 🧪 Event streaming (v3): typed projections. One handle per subagent, by NAME.
stream = make_agent().stream_events(QUESTION, version="v3")
for kind, item in stream.interleave("messages", "subagents"):
    if kind == "messages":
        text = "".join(item.text)                    # iterating .text yields deltas as they arrive
        planned = [c["name"] for c in item.tool_calls.get()]
        print(f"[coordinator] text={text!r} tool_calls={planned}")
    else:
        print(f"[{item.name}] started")
        for message in item.messages:
            print(f"   [{item.name}] text={''.join(message.text)!r} tool_calls={[c['name'] for c in message.tool_calls.get()]}")
        print(f"[{item.name}] {item.status}")
print("final:", stream.output["messages"][-1].text)

**You should now see** the coordinator planning a `task` call, a `[technical]` block with its tool call and answer, the status going from `started` to `completed`, and the coordinator's final text. Handles are **lazy**: a projection opens only when you touch it, so a UI that only shows "technical: working…" costs nothing extra.

> ⚠️ **Production note: iterate projections in the order they arrive.** Projections are fed by one shared pump. Use `interleave(...)` (sync) or `astream_events` + `asyncio.gather` (async) when you consume several at once; reading one projection to the end first can leave another empty.

## Part 3 · UI events and Server-Sent Events

In [ ]:
# 🧪 From raw v3 events to UI events (what a chat front end actually needs), then to Server-Sent Events.
def ui_events(stream):
    for event in stream:
        method, params = event["method"], event["params"]
        source = "subagent" if params["namespace"] else "coordinator"
        if method == "messages":
            payload = params["data"][0]
            if isinstance(payload, dict) and payload.get("event") == "content-block-delta":
                delta = payload.get("delta") or {}
                if delta.get("type") == "text-delta":
                    yield {"type": "token", "source": source, "text": delta["text"]}
        elif method == "tools":
            data = params["data"]
            if data.get("event") == "tool-started":
                yield {"type": "tool", "source": source, "name": data["tool_name"], "input": data["input"]}
            elif data.get("event") == "tool-finished":
                yield {"type": "tool_done", "source": source, "id": data["tool_call_id"]}


events = list(ui_events(make_agent().stream_events(QUESTION, version="v3")))
for e in events:
    print(e)
print("\nas SSE:", repr("data: " + json.dumps(events[0]) + "\n\n"))

**What just happened?** A front end needs three things: tokens (with who is speaking), tool start, tool end. `ui_events()` reduces the raw protocol to exactly that, and one `data: …\n\n` line per event is all Server-Sent Events need. In a FastAPI app you'd wrap it in a `StreamingResponse(media_type="text/event-stream")`.

## Part 4 · Interrupts while streaming

In [ ]:
# 🧪 Streams and interrupts: the stream tells you it paused; resume with a new stream on the same thread.
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


@tool
def issue_refund(invoice_id: str) -> str:
    """Refund an invoice."""
    return f"Refunded {invoice_id}."


script = ScriptedChatModel(responses=[calls(("issue_refund", {"invoice_id": "INV-2001"}, "r")), AIMessage("Done, refunded.")])
agent = build_deep_agent(script, tools=[issue_refund], checkpointer=InMemorySaver(), interrupt_on={"issue_refund": True},
                         fallbacks=False)
config = {"configurable": {"thread_id": "ui-1"}}
stream = agent.stream_events(QUESTION, config, version="v3")
for message in stream.messages:
    "".join(message.text)
print("interrupted:", stream.interrupted, "| waiting on:", [i.value["action_requests"][0]["name"] for i in stream.interrupts])
stream = agent.stream_events(Command(resume={"decisions": [{"type": "approve"}]}), config, version="v3")
print("after approve:", ["".join(m.text) for m in stream.messages], "| interrupted:", stream.interrupted)

**You should now see** `interrupted: True` waiting on `issue_refund`, then the reply after approval. In a UI: show the pending action with its arguments, POST the decision, open a new stream with `Command(resume=…)` on the same `thread_id`.

## Part 5 · 💸 Real tokens

> 💸 Calls your deep model (`MASTERY_DEEP_MODEL`, default Gemini free tier). Rate-limited to one call every 4 s and capped per run. Without the key it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE (opt-in): watch real tokens arrive from the coordinator and a subagent. ≈ 6–10K tokens.
RUN_LIVE = False
if RUN_LIVE and HAVE_DEEP_KEY:
    from mastery.deep import deep_model
    helper = {"name": "explainer", "description": "Explains one concept in two sentences.",
              "system_prompt": "Explain the concept in two sentences.", "model": deep_model("subagent")}
    live = build_deep_agent(subagents=[helper], model_calls=6, tool_calls=4,
                            system_prompt="Delegate the explanation to the explainer subagent, then add one example.")
    stream = live.stream_events({"messages": [{"role": "user", "content": "What is a checkpointer?"}]}, version="v3")
    for e in ui_events(stream):
        if e["type"] == "token":
            print(e["text"], end="", flush=True)
        else:
            print(f"\n[{e['source']}] {e['type']} {e.get('name', '')}")
    print("\nstats:", run_stats(stream.output))
elif not RUN_LIVE:
    print("⏭️ Skipped: set RUN_LIVE = True to try it (≈ 6–10K tokens).")
else:
    print(f"⏭️ Skipped: no key for {DEEP_MODEL} (see the setup cell).")

**You should now see** (when enabled) a `task` tool event, tokens from the subagent, then the coordinator's tokens. On the free tier, the rate limiter (one call every 4 s) shows up as pauses between steps, not inside a message.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| No subagent events | v2 without `subgraphs=True` | Add `subgraphs=True, version="v2"` |
| `LangChainBetaWarning` on every run | v3 is experimental in this LangGraph | Expected; filter it narrowly, as in the helpers |
| Tool call handle has `output=None` | You read it as soon as it started | Read `completed`/`output` after the stream moves on, or use `tool-finished` events |
| Empty projection | Consumed another projection to the end first | `interleave(...)` or async |
| UI shows internal node names | Using `stream.subgraphs` | Use `stream.subagents` for user-facing views |

## 🏋️ Exercises

1. **(Easy)** In Part 1, print the middleware hooks too. Which run before every model call?
2. **(Medium)** Extend `ui_events()` to emit `{"type": "done"}` at the end and `{"type": "interrupt", ...}` when the stream is interrupted.
3. **(Medium)** Use `stream.interleave("messages", "tool_calls")` to print coordinator tool calls as they start.
4. **(Stretch)** Serve `ui_events()` from a FastAPI endpoint as SSE and read it with `curl -N`.

<details>
<summary><b>Show solutions</b></summary>

**1.** Remove the filter: `ModelCallLimitMiddleware.before_model` runs before each model call; `ToolCallLimitMiddleware.after_model` and `ModelCallLimitMiddleware.after_model` after it.

**2.** Add this after the `for` loop inside `ui_events()` (the stream knows whether it paused once you've consumed it):
```python
if stream.interrupted:
    yield {"type": "interrupt", "actions": [i.value["action_requests"] for i in stream.interrupts]}
else:
    yield {"type": "done"}
```

**3.**
```python
s = make_agent().stream_events(QUESTION, version="v3")
for kind, item in s.interleave("messages", "tool_calls"):
    if kind == "tool_calls":
        print("tool:", item.tool_name, item.input)
```

**4.**
```python
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
app = FastAPI()

@app.get("/ask")
def ask(q: str):
    s = make_agent().stream_events({"messages": [{"role": "user", "content": q}]}, version="v3")
    return StreamingResponse((f"data: {json.dumps(e)}\n\n" for e in ui_events(s)), media_type="text/event-stream")
```
</details>

## 🔎 Docs drill (5 min)

For a user-facing progress panel, should you build it on stream.subgraphs or stream.subagents, and why?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Event streaming · Subagents versus subgraphs](https://docs.langchain.com/oss/python/deepagents/event-streaming).
2. `stream.subgraphs` shows graph execution structure (internal nodes).
3. `stream.subagents` shows the delegations users care about, named by `subagent_type`: use it for UI.

</details>

## ✅ Checkpoint

- [ ] I can stream with v2 and identify the source agent by namespace.
- [ ] I can use v3 projections and subagent handles, and I know v3 is still marked experimental.
- [ ] I can map events to tokens/tool start/tool end and send them as SSE.
- [ ] I can pause and resume a streamed run.
- [ ] Restart & Run All passes without keys; outputs cleared before saving.

## 🔎 Research checkpoint

Is the v3 protocol still marked experimental in the latest `langgraph`? Check the release notes and the [Event streaming](https://docs.langchain.com/oss/python/deepagents/event-streaming) page, and log the answer with versions in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- v2 (`subgraphs=True`) is stable; v3 (`stream_events`) is ergonomic and recommended, still experimental here.
- UIs need tokens with a source, tool start/end, interrupts and done: reduce events to that.
- Resume interrupted streams with `Command(resume=…)` on the same thread.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 11-11-evaluating-and-shipping-deep-agents.ipynb** → [`11-11-evaluating-and-shipping-deep-agents.ipynb`](11-11-evaluating-and-shipping-deep-agents.ipynb).

[Module 11 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)